# 02. H1 — план батчей

Для каждого из 300 целевых текстов собираем батчи: целевой + 8 соседей, доля токсичных соседей 0 / 25 / 50 / 75 / 100 %, 3 повтора с разными соседями.
Итого 300 × 5 × 3 = **4 500 батчей**. 

Нужны файлы из ноутбука 01: `data/processed/targets.parquet` и `neighbors_pool.parquet`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
DATA_PROC = PROJECT_DIR / "data" / "processed"
DATA_RES = PROJECT_DIR / "data" / "results"
DATA_RES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

from src.batching import build_h1_plan

targets = pd.read_parquet(DATA_PROC / "targets.parquet")
neighbors = pd.read_parquet(DATA_PROC / "neighbors_pool.parquet")
print(len(targets), "целевых;", len(neighbors), "соседей")

300 целевых; 800 соседей


In [2]:
plan = build_h1_plan(targets, neighbors, n_reps=3, seed=42)
print(len(plan), "батчей")
plan.head()

4500 батчей


,batch_id,target_id,n_toxic,prop_toxic,rep,target_pos,item_ids,length_tol
0,h1_5903407_k0_r0,5903407,0,0.00,0,1,"[5903407, 433520, 5354110, 1049200, 811686, 937573, 6241103, 5846794, 370500]",0.15
1,h1_5903407_k2_r0,5903407,2,0.25,0,1,"[5903407, 811686, 5101525, 5354110, 1049200, 433520, 937573, 543339, 370500]",0.15
2,h1_5903407_k4_r0,5903407,4,0.50,0,1,"[5903407, 5367949, 6259240, 5101525, 370500, 937573, 543339, 5354110, 433520]",0.15
3,h1_5903407_k6_r0,5903407,6,0.75,0,1,"[5903407, 5513610, 5101525, 5354110, 433520, 5260081, 5367949, 6259240, 543339]",0.15
4,h1_5903407_k8_r0,5903407,8,1.00,0,1,"[5903407, 5513610, 543339, 5039567, 5260081, 588916, 5101525, 6259240, 5367949]",0.15


### Проверки плана
1. На каждого целевого ровно 15 батчей (5 долей × 3 повтора).
2. Насколько пришлось расширять допуск по длине (хорошо, если почти везде ±15 %).
3. **Главная проверка:** средняя длина батча почти не зависит от доли токсичных соседей — иначе эффект соседей смешается с эффектом длины.
4. Позиции целевого распределены поровну.

In [3]:
assert plan.groupby("target_id").size().eq(15).all()

print("Допуск по длине:")
print(plan.drop_duplicates(["target_id", "rep"])["length_tol"].value_counts(normalize=True).round(3))

ntok = pd.concat([targets, neighbors]).set_index("id")["n_tokens"]
plan["batch_tokens"] = plan["item_ids"].map(lambda ids: int(ntok.loc[ids].sum()))
print("\nСредняя длина батча (токенов текста) по доле токсичных соседей:")
print(plan.groupby("prop_toxic")["batch_tokens"].agg(["mean", "std"]).round(1))

print("\nПозиции целевого:")
print(plan["target_pos"].value_counts().sort_index())

Допуск по длине:
length_tol
0.15    0.990
0.20    0.007
0.30    0.003
Name: proportion, dtype: float64

Средняя длина батча (токенов текста) по доле токсичных соседей:
             mean    std
prop_toxic              
0.00        353.0  136.9
0.25        353.3  137.3
0.50        353.4  137.3
0.75        353.3  137.4
1.00        353.1  137.4

Позиции целевого:
target_pos
1    1500
5    1500
9    1500
Name: count, dtype: int64


In [4]:
plan.to_parquet(DATA_PROC / "h1_plan.parquet", index=False)
print("Сохранено:", DATA_PROC / "h1_plan.parquet")

Сохранено: d:\statia\data\processed\h1_plan.parquet
